In [1]:
!pip install numpy cupy-cuda12x opencv-python-headless matplotlib
import numpy as np
import cupy as cp
import cv2
import matplotlib.pyplot as plt
from numba import cuda

# Scalar Multiplication of Vector


In [7]:
import numpy as np
from numba import cuda
import time

# 1. Define the CUDA Kernel
@cuda.jit
def scalar_multiply_kernel(d_vector, scalar, d_result):
    # Calculate the global thread index
    idx = cuda.grid(1)

    # Check boundary condition
    if idx < d_vector.size:
        d_result[idx] = d_vector[idx] * scalar

# 2. Generate DS1: 10 Million Floating-Point Values
N = 10_000_000
scalar = 2.5
h_vector = np.random.rand(N).astype(np.float32)
h_result = np.zeros(N).astype(np.float32)

# 3. Allocate device memory and copy data to GPU
d_vector = cuda.to_device(h_vector)
d_result = cuda.to_device(h_result)

# 4. Configure Grid and Block dimensions
threads_per_block = 256
blocks_per_grid = (N + (threads_per_block - 1)) // threads_per_block

# 5. Launch the kernel and time it
start_time = time.time()
scalar_multiply_kernel[blocks_per_grid, threads_per_block](d_vector, scalar, d_result)
cuda.synchronize() # Wait for GPU to finish
end_time = time.time()

# 6. Copy result back to Host
h_result = d_result.copy_to_host()

# 7. Verification & Output
print(u"--- Task 1: DS1 Vector Operations ---")
print(f"Vector Size: {N:,} elements")
print(f"Execution Time on GPU: {(end_time - start_time) * 1000:.2f} ms")
print(f"Validation (First 5 elements):")
print(f"Original: {h_vector[:5]}")
print(f"Expected: {h_vector[:5] * scalar}")
print(f"GPU Result: {h_result[:5]}")
assert np.allclose(h_result, h_vector * scalar), "Verification Failed!"
print("Verification SUCCESS!")

--- Task 1: DS1 Vector Operations ---
Vector Size: 10,000,000 elements
Execution Time on GPU: 53.30 ms
Validation (First 5 elements):
Original: [0.1181298  0.29644522 0.4672611  0.7080555  0.72392195]
Expected: [0.2953245  0.74111307 1.1681528  1.7701387  1.8098049 ]
GPU Result: [0.2953245  0.74111307 1.1681528  1.7701387  1.8098049 ]
Verification SUCCESS!
